In [5]:
import os
import sys
from pathlib import Path

import torch
import pandas as pd

PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from preprocess import DemuxManager
from classifier import HTClassifier
from evaluator import evaluate_demux
manager = DemuxManager()

Project root: /home/chongxiao/HT_Demux


In [6]:

DATA_DIR = PROJECT_ROOT / "HTO016"

mtx_path  = DATA_DIR / "matrix.mtx.gz"
feat_path = DATA_DIR / "features.tsv.gz"
bar_path  = DATA_DIR / "barcodes.tsv.gz"

assert mtx_path.exists(), f"Missing {mtx_path}"
assert feat_path.exists(), f"Missing {feat_path}"
assert bar_path.exists(), f"Missing {bar_path}"

print("Data directory:", DATA_DIR)


Data directory: /home/chongxiao/HT_Demux/HTO016


In [7]:
print("Loading data...")
counts, barcodes = manager.load_10x_mtx(
    mtx_path,
    feat_path,
    bar_path
)

manager.build_configs(max_klet=2)

print("Counts shape:", counts.shape)
print("Number of barcodes:", len(barcodes))


Loading data...
Counts shape: torch.Size([197926, 16])
Number of barcodes: 197926


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Running optimizer on {device}...")
results = manager.run_demux(
    counts,
    model_type="GMM",     # or "NB"
    method="GD",          # or "EM"
    device=device
)

Running optimizer on cuda...


In [9]:
clf = HTClassifier(
    cfg_labels=results["cfg_labels"],
    tag_names=results["tag_names"],
    configs=manager.configs
)

df_results = clf.classify(
    posteriors=results["posteriors"],
    barcodes=barcodes,
    map_thresh=0.1
)

df_results.head()

,barcode,assignment_raw,assignment,assignment_final,probability,p_HTO_sim_01,p_HTO_sim_02,p_HTO_sim_03,p_HTO_sim_04,p_HTO_sim_05,...,p_HTO_sim_07,p_HTO_sim_08,p_HTO_sim_09,p_HTO_sim_10,p_HTO_sim_11,p_HTO_sim_12,p_HTO_sim_13,p_HTO_sim_14,p_HTO_sim_15,p_HTO_sim_16
0,droplet_00001,HTO_sim_03,HTO_sim_03,HTO_sim_03,0.893783,0.091119,6.978912e-05,1.000000,2.843285e-03,0.001372,...,0.000025,0.004024,3.100631e-06,0.000370,0.005807,0.000108,0.000170,0.000082,0.000224,1.204394e-07
1,droplet_00008,HTO_sim_11,HTO_sim_11,HTO_sim_11,0.560761,0.000637,2.572633e-08,0.001695,7.175995e-04,0.003955,...,0.000002,0.012429,3.720474e-04,0.417184,0.998317,0.001806,0.000140,0.000085,0.000002,1.264552e-05
2,droplet_00010,HTO_sim_14+HTO_sim_15,Multiplet,Multiplet,0.684112,0.000078,2.452988e-05,0.000010,1.204753e-05,0.000673,...,0.000011,0.000076,9.612671e-07,0.000005,0.000154,0.000130,0.000195,1.000000,0.684113,1.130646e-06
3,droplet_00014,HTO_sim_05,HTO_sim_05,HTO_sim_05,0.973508,0.000052,1.465397e-06,0.000013,1.490486e-05,1.000000,...,0.000295,0.001612,2.399693e-04,0.000128,0.001750,0.001097,0.014276,0.003596,0.003410,6.133130e-07
4,droplet_00015,HTO_sim_11,HTO_sim_11,HTO_sim_11,0.801963,0.000350,7.302270e-05,0.004294,2.881415e-08,0.000002,...,0.000457,0.006274,1.956207e-04,0.011903,0.999565,0.000058,0.154662,0.000011,0.019597,1.738179e-07


In [10]:

OUT_DIR = PROJECT_ROOT / "results"
OUT_DIR.mkdir(exist_ok=True)

pred_path = OUT_DIR / "demux_results.csv"
df_results.to_csv(pred_path, index=False)

print(f"Demux complete. Results saved to {pred_path}")


Demux complete. Results saved to /home/chongxiao/HT_Demux/results/demux_results.csv


In [11]:
truth_path = DATA_DIR / "ground_truth_droplet.csv"
assert truth_path.exists(), f"Missing {truth_path}"

evaluate_demux(
    pred_path=pred_path,
    truth_path=truth_path
)


      HT-Demux Performance Evaluation
Total droplets matched: 197926
Overall Accuracy: 81.27%

Classification Report:
              precision    recall  f1-score   support

  HTO_sim_01       0.91      0.80      0.85     10924
  HTO_sim_02       0.92      0.85      0.88     11020
  HTO_sim_03       0.88      0.80      0.84     10973
  HTO_sim_04       0.93      0.82      0.87     11099
  HTO_sim_05       0.90      0.84      0.87     11109
  HTO_sim_06       0.95      0.86      0.90     11190
  HTO_sim_07       0.91      0.84      0.88     11038
  HTO_sim_08       0.88      0.83      0.86     10908
  HTO_sim_09       0.91      0.82      0.86     11179
  HTO_sim_10       0.89      0.84      0.87     10981
  HTO_sim_11       0.87      0.85      0.86     11186
  HTO_sim_12       0.89      0.80      0.84     11144
  HTO_sim_13       0.87      0.83      0.85     11103
  HTO_sim_14       0.92      0.82      0.87     10866
  HTO_sim_15       0.87      0.84      0.86     11050
  HTO_sim_16    